# Module 4: Open Quantum Systems, Dissipation & Trajectories
## Notebook 4.6: Input-Output Theory & Cavity Scattering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_4_open_systems_dissipation/08_input_output_theory_and_cavity_scattering.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** Light-Matter Interaction & Cavity QED
* **Module 4:** **Open Quantum Systems, Dissipation & Trajectories**
  * **4.1** Density Matrix, Purity & Entanglement Entropy
  * **4.2** The Lindblad Master Equation & Dynamical Semigroups
  * **4.3** Cavity Decay, Photon Loss & Thermal Reservoirs
  * **4.4** The Monte Carlo Wavefunction (Quantum Jump) Method
  * **4.5** Quantum Trajectories & Homodyne Detection Unraveling
  * **4.6 Input-Output Theory & Cavity Scattering** $\leftarrow$ *(You are here)*
  * **4.7** Laser Theory, Micromaser & Trapping States

---

### Learning Objectives
1. **The Gardiner-Collett Formalism:** Derive the fundamental boundary condition $\hat{a}_{\text{out}}(t) + \hat{a}_{\text{in}}(t) = \sqrt{\kappa}\hat{a}(t)$ linking propagating continuous traveling-wave modes to localized discrete cavity modes.
2. **Heisenberg-Langevin Equations:** Solve operator equations of motion in the frequency domain, incorporating coherent input drives, internal losses, and external radiation channels.
3. **Scattering Regimes & Impedance Matching:** Characterize reflection spectra $r(\omega)$ across the undercoupled ($\kappa_c < \kappa_i$), critically coupled ($\kappa_c = \kappa_i$, $r(\omega_c) = 0$), and overcoupled ($\kappa_c \gg \kappa_i$) regimes.
4. **Complex Plane & Smith Chart Geometries:** Track reflection trajectories in the complex plane $\text{Re}[r]$ vs. $\text{Im}[r]$, revealing the phase-slip topology across resonance.
5. **Cavity QED Transmission & Purcell Effect:** Model the transmission of coupled atom-cavity systems, demonstrating vacuum Rabi splitting in transmission and computing the Purcell enhancement factor $F_P = \frac{4g^2}{\kappa \gamma}$.



In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import math
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "lines.linewidth": 1.8,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

# Verify environment capabilities
print(f"NumPy version: {np.__version__}")
try:
    import qutip as qt
    print(f"QuTiP version: {qt.__version__} (Available for optional comparison)")
    HAS_QUTIP = True
except ImportError:
    print("QuTiP not installed; running pure, optimized native NumPy/SciPy engine.")
    HAS_QUTIP = False



---

## Theoretical Framework: The Gardiner-Collett Input-Output Formalism

### 1. Connecting Discrete Cavities to Propagating Fields
In realistic quantum optical experiments—such as photonic crystal cavities, optical Fabry-Pérot resonators, and superconducting circuit QED transmission lines—systems are driven and probed by propagating traveling-wave radiation.
The **Gardiner-Collett Input-Output formalism** (1985) rigorously bridges localized, discrete bound modes with continuous external baths of electromagnetic radiation.

Consider an internal cavity mode $\hat{a}$ coupled to a continuum of external bath modes $\hat{b}(\omega)$ through a semi-transparent mirror with coupling rate $\kappa$:
$$\hat{H} = \hat{H}_{\text{sys}} + \int_{-\infty}^{\infty} d\omega \, \hbar \omega \, \hat{b}^\dagger(\omega)\hat{b}(\omega) + i\hbar \sqrt{\frac{\kappa}{2\pi}} \int_{-\infty}^{\infty} d\omega \left[ \hat{b}^\dagger(\omega)\hat{a} - \hat{a}^\dagger \hat{b}(\omega) \right]$$

We define the propagating **input field** $\hat{a}_{\text{in}}(t)$ and **output field** $\hat{a}_{\text{out}}(t)$ in the time domain:
$$\hat{a}_{\text{in}}(t) \equiv -\frac{1}{\sqrt{2\pi}} \int_{-\infty}^{\infty} d\omega \, e^{-i\omega(t - t_0)} \hat{b}(t_0, \omega) \quad (t_0 < t)$$
$$\hat{a}_{\text{out}}(t) \equiv \frac{1}{\sqrt{2\pi}} \int_{-\infty}^{\infty} d\omega \, e^{-i\omega(t - t_1)} \hat{b}(t_1, \omega) \quad (t_1 > t)$$
where $[\hat{a}_{\text{in}}(t), \hat{a}_{\text{in}}^\dagger(t')] = \delta(t - t')$.

---

### 2. The Heisenberg-Langevin Equation & Boundary Condition
Integrating the Heisenberg equation of motion for the bath operators yields the **Heisenberg-Langevin equation** for the cavity mode:
$$\frac{d\hat{a}}{dt} = -\frac{i}{\hbar}[\hat{a}, \hat{H}_{\text{sys}}] - \frac{\kappa_{\text{tot}}}{2}\hat{a}(t) - \sum_k \sqrt{\kappa_k}\hat{a}_{\text{in}, k}(t)$$
and the celebrated **Gardiner-Collett boundary condition**:
$$\hat{a}_{\text{out}, k}(t) = \hat{a}_{\text{in}, k}(t) + \sqrt{\kappa_k}\hat{a}(t)$$

---

### 3. One-Sided Cavity: Reflection & Absorption
Consider a single-port resonator with external coupling rate $\kappa_c$ and internal parasitic loss rate $\kappa_i$ ($\kappa_{\text{tot}} = \kappa_c + \kappa_i$).
In the rotating frame with detuning $\Delta = \omega - \omega_c$:
$$-i\omega \hat{a}(\omega) = -\left(i\omega_c + \frac{\kappa_c + \kappa_i}{2}\right)\hat{a}(\omega) - \sqrt{\kappa_c}\hat{a}_{\text{in}}(\omega)$$
$$\hat{a}(\omega) = \frac{-\sqrt{\kappa_c}}{i(\omega_c - \omega) + \frac{\kappa_c + \kappa_i}{2}}\hat{a}_{\text{in}}(\omega)$$

Substituting into the boundary condition $\hat{a}_{\text{out}}(\omega) = \hat{a}_{\text{in}}(\omega) + \sqrt{\kappa_c}\hat{a}(\omega)$, the complex reflection coefficient $r(\omega) \equiv \hat{a}_{\text{out}}(\omega) / \hat{a}_{\text{in}}(\omega)$ is:
$$r(\omega) = 1 - \frac{\kappa_c}{i(\omega_c - \omega) + \frac{\kappa_c + \kappa_i}{2}} = \frac{i(\omega - \omega_c) + \frac{\kappa_c - \kappa_i}{2}}{i(\omega - \omega_c) + \frac{\kappa_c + \kappa_i}{2}}$$

#### The Three Fundamental Coupling Regimes:
1. **Undercoupled ($\kappa_c < \kappa_i$):** Internal dissipation dominates over external coupling. On resonance ($\omega = \omega_c$), $r(\omega_c) = \frac{\kappa_c - \kappa_i}{\kappa_c + \kappa_i} < 0$. The reflection phase does not encircle the origin in the complex plane.
2. **Critically Coupled ($\kappa_c = \kappa_i$):** External coupling exactly matches internal losses (impedance matching). On resonance:
   $$r(\omega_c) = 0 \implies R(\omega_c) = |r(\omega_c)|^2 = 0, \quad A(\omega_c) = 1.0$$
   **100% of incident optical power is absorbed by the cavity!**
3. **Overcoupled ($\kappa_c \gg \kappa_i$):** External coupling dominates. On resonance $r(\omega_c) \approx +1$. The complex reflection vector traces a full $2\pi$ circle enclosing the origin on the Smith chart.

---

### 4. Two-Sided Cavity: Transmission & Finesse
For a Fabry-Pérot cavity with input mirror 1 ($\kappa_1$), output mirror 2 ($\kappa_2$), and internal loss $\kappa_i$:
$$r(\omega) = \frac{i(\omega - \omega_c) + \frac{\kappa_1 - \kappa_2 - \kappa_i}{2}}{i(\omega - \omega_c) + \frac{\kappa_1 + \kappa_2 + \kappa_i}{2}}, \quad t(\omega) = \frac{\sqrt{\kappa_1 \kappa_2}}{i(\omega - \omega_c) + \frac{\kappa_1 + \kappa_2 + \kappa_i}{2}}$$
Energy conservation rigorously demands:
$$R(\omega) + T(\omega) + A(\omega) = |r(\omega)|^2 + |t(\omega)|^2 + A(\omega) \equiv 1.0$$

---

### 5. Atom-Cavity Scattering & The Purcell Effect
When a two-level atom (transition frequency $\omega_a$, decay rate $\gamma$) is coupled to the cavity mode with coupling strength $g$, the transmission coefficient becomes:
$$t(\omega) = \frac{\sqrt{\kappa_1 \kappa_2}}{i(\omega_c - \omega) + \frac{\kappa_{\text{tot}}}{2} + \frac{g^2}{i(\omega_a - \omega) + \frac{\gamma}{2}}}$$

* **Strong Coupling Regime ($g > \kappa/4, \gamma/4$):** The transmission resonance splits into two dressed peaks separated by the **vacuum Rabi splitting** $\Delta\omega = 2g$.
* **Bad-Cavity / Purcell Regime ($\kappa \gg g \gg \gamma$):** Spontaneous emission into the cavity mode is accelerated by the **Purcell factor**:
  $$F_P = \frac{4g^2}{\kappa \gamma}$$



In [ ]:
# Implementation: High-Precision Cavity Input-Output Scattering Engine

class CavityScatteringEngine:
    """
    Input-Output Scattering Engine for single-mode optical and microwave resonators.
    Calculates reflection, transmission, absorption, and atom-cavity coupled spectra.
    """
    def __init__(self, omega_c=0.0, kappa_c=1.0, kappa_i=0.2):
        self.omega_c = float(omega_c)
        self.kappa_c = float(kappa_c)
        self.kappa_i = float(kappa_i)
        self.kappa_tot = self.kappa_c + self.kappa_i

    def reflection_one_sided(self, omega_array, kappa_c=None, kappa_i=None):
        """
        Compute complex reflection coefficient r(omega) for a one-sided cavity:
            r(omega) = [i*(omega - omega_c) + (kappa_c - kappa_i)/2] /
                       [i*(omega - omega_c) + (kappa_c + kappa_i)/2]
        """
        kc = self.kappa_c if kappa_c is None else float(kappa_c)
        ki = self.kappa_i if kappa_i is None else float(kappa_i)
        
        delta = omega_array - self.omega_c
        num = 1j * delta + 0.5 * (kc - ki)
        denom = 1j * delta + 0.5 * (kc + ki)
        r = num / denom
        
        R = np.abs(r)**2
        A = 1.0 - R  # Parasitic internal absorption
        phase = np.angle(r)
        
        return {"r": r, "R": R, "A": A, "phase": phase}

    def scattering_two_sided(self, omega_array, kappa_1=1.0, kappa_2=1.0, kappa_i=0.0):
        """
        Compute complex reflection r(omega) and transmission t(omega) for a two-sided cavity.
        """
        delta = omega_array - self.omega_c
        k_tot = kappa_1 + kappa_2 + kappa_i
        
        num_r = 1j * delta + 0.5 * (kappa_1 - kappa_2 - kappa_i)
        denom = 1j * delta + 0.5 * k_tot
        
        r = num_r / denom
        t = np.sqrt(kappa_1 * kappa_2) / denom
        
        R = np.abs(r)**2
        T = np.abs(t)**2
        A = 1.0 - (R + T)
        
        return {"r": r, "t": t, "R": R, "T": T, "A": A}

    def atom_cavity_transmission(self, omega_array, g=1.5, gamma=0.1, kappa_1=0.5, kappa_2=0.5, kappa_i=0.0):
        """
        Compute transmission of an atom-cavity system exhibiting vacuum Rabi splitting:
            t(omega) = sqrt(k1 * k2) / [ i*(omega_c - omega) + k_tot/2 + g^2 / (i*(omega_a - omega) + gamma/2) ]
        """
        k_tot = kappa_1 + kappa_2 + kappa_i
        delta_c = self.omega_c - omega_array
        delta_a = self.omega_c - omega_array  # Atom tuned on resonance with cavity
        
        atom_susceptibility = (g**2) / (1j * delta_a + 0.5 * gamma)
        denom = 1j * delta_c + 0.5 * k_tot + atom_susceptibility
        t = np.sqrt(kappa_1 * kappa_2) / denom
        
        return {"t": t, "T": np.abs(t)**2, "phase": np.angle(t)}

print("CavityScatteringEngine successfully loaded.")



In [ ]:
# Publication-Quality 4-Panel Visualization: Input-Output Spectra, Smith Chart & Vacuum Rabi Splitting

engine = CavityScatteringEngine(omega_c=0.0)

# Detuning grids
omega_grid = np.linspace(-4.0, 4.0, 1001)
omega_rabi_grid = np.linspace(-6.0, 6.0, 1501)

# 1. One-sided reflection across the 3 regimes
# Undercoupled: kc = 0.2, ki = 1.0 (kc < ki)
res_under = engine.reflection_one_sided(omega_grid, kappa_c=0.2, kappa_i=1.0)
# Critically coupled: kc = 1.0, ki = 1.0 (kc = ki)
res_crit = engine.reflection_one_sided(omega_grid, kappa_c=1.0, kappa_i=1.0)
# Overcoupled: kc = 3.0, ki = 0.2 (kc >> ki)
res_over = engine.reflection_one_sided(omega_grid, kappa_c=3.0, kappa_i=0.2)

# 2. Two-sided symmetric cavity
res_2side = engine.scattering_two_sided(omega_grid, kappa_1=1.0, kappa_2=1.0, kappa_i=0.2)

# 3. Atom-cavity vacuum Rabi splitting
res_atom_0 = engine.atom_cavity_transmission(omega_rabi_grid, g=0.0, kappa_1=0.5, kappa_2=0.5)
res_atom_1 = engine.atom_cavity_transmission(omega_rabi_grid, g=1.5, kappa_1=0.5, kappa_2=0.5)
res_atom_2 = engine.atom_cavity_transmission(omega_rabi_grid, g=3.0, kappa_1=0.5, kappa_2=0.5)

# Generate 4-Panel Masterpiece Figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# -------------------------------------------------------------
# Panel (a): One-Sided Cavity Reflection & Critical Coupling
# -------------------------------------------------------------
ax_a = axes[0, 0]
ax_a.plot(omega_grid, res_under["R"], color="#3498db", linewidth=2.0, label=r"Undercoupled ($\kappa_c = 0.2\kappa_i$)")
ax_a.plot(omega_grid, res_crit["R"], color="#e74c3c", linewidth=2.4, label=r"Critically Coupled ($\kappa_c = \kappa_i$)")
ax_a.plot(omega_grid, res_over["R"], color="#2ecc71", linewidth=2.0, label=r"Overcoupled ($\kappa_c = 15\kappa_i$)")
ax_a.plot(omega_grid, res_crit["A"], color="black", linestyle=":", linewidth=2.0, label=r"Critical Absorption $A(\omega)$")

ax_a.scatter([0.0], [0.0], color="#e74c3c", marker="o", s=80, zorder=5)
ax_a.annotate("Impedance Match: " + r"$R(\omega_c) = 0$" + "\n" + r"$100\%$ Absorption", 
              xy=(0.0, 0.0), xytext=(0.4, 0.35),
              arrowprops=dict(arrowstyle="->", color="#e74c3c", lw=1.5),
              bbox=dict(boxstyle="round,pad=0.4", fc="#fadbd8", ec="#e74c3c", lw=1.2),
              fontweight="bold")

ax_a.set_title(r"(a) Reflection & Absorption in One-Sided Cavity", fontweight="bold")
ax_a.set_xlabel(r"Probe Detuning $(\omega - \omega_c) / \kappa_{\rm ref}$")
ax_a.set_ylabel(r"Power Ratio ($R, A$)")
ax_a.set_ylim(-0.05, 1.05)
ax_a.legend(loc="center right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (b): Complex Plane Trajectory (Smith / Nyquist Chart)
# -------------------------------------------------------------
ax_b = axes[0, 1]
ax_b.plot(np.real(res_under["r"]), np.imag(res_under["r"]), color="#3498db", linewidth=2.0, label="Undercoupled")
ax_b.plot(np.real(res_crit["r"]), np.imag(res_crit["r"]), color="#e74c3c", linewidth=2.4, label="Critically Coupled")
ax_b.plot(np.real(res_over["r"]), np.imag(res_over["r"]), color="#2ecc71", linewidth=2.0, label="Overcoupled")

# Unit circle & axes
theta_circle = np.linspace(0, 2*np.pi, 200)
ax_b.plot(np.cos(theta_circle), np.sin(theta_circle), "k--", alpha=0.4, label="Unit Circle $|r|=1$")
ax_b.scatter([0.0], [0.0], color="black", marker="+", s=100, label="Origin $(0,0)$", zorder=5)
ax_b.scatter([1.0], [0.0], color="black", marker="s", s=60, label=r"Off-Resonance $r(\pm\infty)=1$", zorder=5)

ax_b.set_title(r"(b) Complex Reflection Trajectory: $\text{Re}[r]$ vs. $\text{Im}[r]$", fontweight="bold")
ax_b.set_xlabel(r"Real Part $\text{Re}[r(\omega)]$")
ax_b.set_ylabel(r"Imaginary Part $\text{Im}[r(\omega)]$")
ax_b.set_xlim(-1.15, 1.15)
ax_b.set_ylim(-1.15, 1.15)
ax_b.set_aspect("equal")
ax_b.legend(loc="lower left", framealpha=0.85)

# -------------------------------------------------------------
# Panel (c): Two-Sided Fabry-Pérot Cavity Scattering
# -------------------------------------------------------------
ax_c = axes[1, 0]
ax_c.plot(omega_grid, res_2side["T"], color="#27ae60", linewidth=2.2, label=r"Transmission $T(\omega) = |t|^2$")
ax_c.plot(omega_grid, res_2side["R"], color="#2980b9", linewidth=2.0, label=r"Reflection $R(\omega) = |r|^2$")
ax_c.plot(omega_grid, res_2side["A"], color="#e67e22", linestyle="--", linewidth=1.8, label=r"Internal Loss $A(\omega)$")
ax_c.plot(omega_grid, res_2side["T"] + res_2side["R"] + res_2side["A"], color="black", linestyle=":", linewidth=2.2, label=r"Sum $R + T + A \equiv 1$")

ax_c.set_title(r"(c) Two-Sided Resonator: $R(\omega), T(\omega), A(\omega)$", fontweight="bold")
ax_c.set_xlabel(r"Probe Detuning $(\omega - \omega_c) / \kappa_1$")
ax_c.set_ylabel(r"Scattering Probability")
ax_c.set_ylim(-0.05, 1.08)
ax_c.legend(loc="center right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (d): Atom-Cavity Vacuum Rabi Splitting in Transmission
# -------------------------------------------------------------
ax_d = axes[1, 1]
ax_d.plot(omega_rabi_grid, res_atom_0["T"], color="#7f8c8d", linestyle="--", linewidth=1.8, label=r"Empty Cavity ($g=0$)")
ax_d.plot(omega_rabi_grid, res_atom_1["T"], color="#9b59b6", linewidth=2.0, label=r"Coupled $g = 1.5\kappa$")
ax_d.plot(omega_rabi_grid, res_atom_2["T"], color="#e74c3c", linewidth=2.4, label=r"Strong Coupling $g = 3.0\kappa$")

# Mark peak locations for g = 3.0
ax_d.axvline(-3.0, color="#e74c3c", linestyle=":", alpha=0.6)
ax_d.axvline(3.0, color="#e74c3c", linestyle=":", alpha=0.6)
ax_d.annotate(r"Vacuum Rabi Splitting $\Delta\omega = 2g = 6\kappa$", 
              xy=(0.0, 0.25), xytext=(-2.8, 0.65),
              arrowprops=dict(arrowstyle="<->", color="black", lw=1.5),
              bbox=dict(boxstyle="round,pad=0.4", fc="#f4ecf7", ec="#9b59b6", lw=1.2),
              fontweight="bold")

ax_d.set_title(r"(d) Vacuum Rabi Splitting in Cavity Transmission", fontweight="bold")
ax_d.set_xlabel(r"Probe Detuning $(\omega - \omega_c) / \kappa$")
ax_d.set_ylabel(r"Transmission $T(\omega)$")
ax_d.set_ylim(-0.05, 1.05)
ax_d.legend(loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()



> **Physical Insight: Impedance Matching, Phase Slips, and Cavity QED Transmission Dips**
>
> 1. **Why Does Critical Coupling Extinguish Reflection?** In a one-sided cavity, the total reflected field $\hat{a}_{\text{out}}$ consists of two interfering pathways: (i) the promptly reflected beam off the front mirror, and (ii) the cavity leakage field leaking back out through the front mirror. On resonance, these two pathways are exactly $180^\circ$ out of phase! When $\kappa_c = \kappa_i$, their amplitudes are precisely identical, resulting in complete destructive interference: $r(\omega_c) = 0$. Every single incident photon enters the cavity and is absorbed by internal dissipation!
> 2. **Phase Slips in the Complex Plane:** 
>    * In the **undercoupled** regime ($\kappa_c < \kappa_i$), the origin lies outside the circular trajectory; as the probe sweeps through resonance, the reflection phase $\arg(r)$ oscillates but returns to zero without circling $2\pi$.
>    * In the **overcoupled** regime ($\kappa_c > \kappa_i$), the trajectory encloses the origin $(0, 0)$; the phase undergoes a complete $2\pi$ phase slip, which forms the physical basis of microwave dispersive readout in circuit QED.
> 3. **Vacuum Rabi Splitting in Transmission:** When an atom is placed on resonance inside the cavity, the strong Jaynes-Cummings dipole coupling $g(\hat{a}^\dagger\hat{\sigma}_- + \hat{a}\hat{\sigma}_+)$ hybridizes the cavity photon and atomic excitation into the dressed states $|1, \pm\rangle = \frac{|1, g\rangle \pm |0, e\rangle}{\sqrt{2}}$. A probe laser tuned to the bare cavity resonance $\omega_c$ cannot excite the system because neither dressed state exists at $\omega_c$, causing a transmission blackout at zero detuning and generating the twin vacuum Rabi peaks at $\omega = \omega_c \pm g$.



In [ ]:
# Interactive Exploration: Cavity Scattering Regimes & Atom-Cavity Splitting

def interactive_scattering_explorer(coupling_ratio=1.0, g_over_kappa=2.0, show_atom=True):
    """
    Interactive exploration of reflection/transmission spectra and vacuum Rabi splitting.
    """
    engine = CavityScatteringEngine(omega_c=0.0)
    omega_grid = np.linspace(-5.0, 5.0, 1001)
    
    # Calculate one-sided reflection with kappa_i = 1.0, kappa_c = coupling_ratio
    res_refl = engine.reflection_one_sided(omega_grid, kappa_c=coupling_ratio, kappa_i=1.0)
    
    # Calculate atom-cavity transmission
    res_trans = engine.atom_cavity_transmission(omega_grid, g=g_over_kappa, kappa_1=0.5, kappa_2=0.5)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # Subplot 1: Reflection & Absorption
    ax1.plot(omega_grid, res_refl["R"], color="#e74c3c", linewidth=2.2, label=r"Reflection $R(\omega)$")
    ax1.plot(omega_grid, res_refl["A"], color="#2c3e50", linewidth=1.8, linestyle="--", label=r"Absorption $A(\omega)$")
    ax1.set_title(r"One-Sided Cavity ($\kappa_c/\kappa_i = " + f"{coupling_ratio:.2f}" + r"$)")
    ax1.set_xlabel(r"$(\omega - \omega_c) / \kappa_i$")
    ax1.set_ylabel("Power Fraction")
    ax1.set_ylim(-0.05, 1.05)
    ax1.legend(loc="center right")
    
    # Subplot 2: Atom-Cavity Transmission
    if show_atom:
        lbl_atom = r"Atom-Cavity $T(\omega)$ ($g = " + f"{g_over_kappa:.1f}" + r"\kappa$)"
        ax2.plot(omega_grid, res_trans["T"], color="#8e44ad", linewidth=2.2, label=lbl_atom)
        ax2.set_title("Atom-Cavity Vacuum Rabi Transmission")
    else:
        res_empty = engine.scattering_two_sided(omega_grid, kappa_1=0.5, kappa_2=0.5, kappa_i=0.0)
        ax2.plot(omega_grid, res_empty["T"], color="#27ae60", linewidth=2.2, label="Empty Cavity Transmission")
        ax2.set_title("Empty Cavity Transmission")
        
    ax2.set_xlabel(r"$(\omega - \omega_c) / \kappa$")
    ax2.set_ylabel("Transmission $T(\omega)$")
    ax2.set_ylim(-0.05, 1.05)
    ax2.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()

# Test invocation
interactive_scattering_explorer(coupling_ratio=1.0, g_over_kappa=2.0, show_atom=True)

try:
    import ipywidgets as widgets
    from IPython.display import display
    w = widgets.interactive(
        interactive_scattering_explorer,
        coupling_ratio=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description=r"$\kappa_c/\kappa_i$"),
        g_over_kappa=widgets.FloatSlider(value=2.0, min=0.0, max=5.0, step=0.25, description=r"$g/\kappa$"),
        show_atom=widgets.Checkbox(value=True, description="Couple Atom")
    )
    display(w)
except Exception:
    pass



---

## Guided Computational Challenges & Exercises

### Challenge 1: Proof of Strict Energy Conservation
1. For an arbitrary two-sided dissipative resonator with coupling rates $\kappa_1, \kappa_2$ and internal loss rate $\kappa_i$, prove algebraically from the Gardiner-Collett equations that:
   $$R(\omega) + T(\omega) + A(\omega) \equiv 1.0 \quad \forall \omega \in \mathbb{R}$$
2. Verify numerically across an extensive grid of 50,000 frequencies that the residual deviation $|R + T + A - 1|$ is bounded by machine precision ($< 10^{-14}$).

---

### Challenge 2: Critical Coupling Impedance Matching
1. Show analytically that the resonant reflection coefficient $r(\omega_c)$ vanishes if and only if $\kappa_c = \kappa_i$:
   $$r(\omega_c) = \frac{\kappa_c - \kappa_i}{\kappa_c + \kappa_i} = 0 \iff \kappa_c = \kappa_i$$
2. Benchmark this extinction condition: assert that on resonance, the reflected power satisfies $R(\omega_c) < 10^{-15}$ and the absorbed power reaches $A(\omega_c) = 1.0 \pm 10^{-15}$.

---

### Challenge 3: Vacuum Rabi Splitting Precision Benchmark
1. In the strong coupling regime ($g \gg \kappa, \gamma$), show that the two transmission maxima of an atom-cavity system occur at frequencies:
   $$\omega_{\pm} = \omega_c \pm \sqrt{g^2 - \frac{(\kappa - \gamma)^2}{16}} \approx \omega_c \pm g$$
2. For $g = 4.0\kappa$ and $\gamma = 0.05\kappa$, locate the two transmission peaks numerically and assert that their separation matches $2g$ to within $0.1\%$.



In [ ]:
# Numerical Verification of Challenges 1, 2, and 3: Input-Output Scattering Benchmarks

engine = CavityScatteringEngine(omega_c=0.0)

# -------------------------------------------------------------
# Part 1: Strict Energy Conservation (Challenge 1)
# -------------------------------------------------------------
omega_dense = np.linspace(-100.0, 100.0, 50000)
k1_test, k2_test, ki_test = 1.5, 0.8, 0.3
res_energy = engine.scattering_two_sided(omega_dense, kappa_1=k1_test, kappa_2=k2_test, kappa_i=ki_test)

total_power = res_energy["R"] + res_energy["T"] + res_energy["A"]
max_energy_deviation = np.max(np.abs(total_power - 1.0))
print(f"Energy Conservation Check (50,000 frequencies):")
print(f"- Max deviation |R + T + A - 1|: {max_energy_deviation:.2e}")
assert max_energy_deviation < 1e-13, f"Energy conservation violated: dev = {max_energy_deviation:.2e} >= 1e-13!"

# -------------------------------------------------------------
# Part 2: Critical Coupling Zero-Reflection (Challenge 2)
# -------------------------------------------------------------
res_crit = engine.reflection_one_sided(np.array([0.0]), kappa_c=2.5, kappa_i=2.5)
R_crit = res_crit["R"][0]
A_crit = res_crit["A"][0]

print(f"\nCritical Coupling Benchmark (kappa_c = kappa_i = 2.5):")
print(f"- Reflected Power R(omega_c): {R_crit:.2e} (Expected: 0.0)")
print(f"- Absorbed Power A(omega_c):  {A_crit:.6f} (Expected: 1.0)")
assert np.isclose(R_crit, 0.0, atol=1e-15), "Critical coupling failed to extinguish reflection!"
assert np.isclose(A_crit, 1.0, atol=1e-15), "Critical coupling failed to achieve 100% absorption!"

# -------------------------------------------------------------
# Part 3: Vacuum Rabi Splitting Separation Benchmark (Challenge 3)
# -------------------------------------------------------------
g_test = 4.0
gamma_test = 0.05
k1_rabi, k2_rabi = 0.5, 0.5
res_rabi = engine.atom_cavity_transmission(omega_dense, g=g_test, gamma=gamma_test, kappa_1=k1_rabi, kappa_2=k2_rabi)

# Find peak locations in transmission
T_vals = res_rabi["T"]
# Split domain into negative and positive frequencies
neg_mask = omega_dense < 0.0
pos_mask = omega_dense > 0.0

peak_neg = omega_dense[neg_mask][np.argmax(T_vals[neg_mask])]
peak_pos = omega_dense[pos_mask][np.argmax(T_vals[pos_mask])]
splitting = peak_pos - peak_neg
expected_splitting = 2.0 * g_test

print(f"\nVacuum Rabi Splitting Benchmark (g = {g_test}):")
print(f"- Lower Peak:     {peak_neg:.4f}")
print(f"- Upper Peak:     {peak_pos:.4f}")
print(f"- Peak Splitting: {splitting:.4f} (Expected: 2g = {expected_splitting:.4f})")
print(f"- Discrepancy:    {np.abs(splitting - expected_splitting):.4f}")

assert np.isclose(splitting, expected_splitting, rtol=1e-3), "Vacuum Rabi splitting deviates by > 0.1%!"

print("=" * 70)
print("SUCCESS: Input-Output Theory & Cavity Scattering Numerically Verified!")
print(f"- Energy conservation: |R + T + A - 1| = {max_energy_deviation:.2e} (PASS: < 1e-13)")
print(f"- Critical coupling:   R(omega_c) = {R_crit:.2e}, A = {A_crit:.2f} (PASS)")
print(f"- Vacuum Rabi split:   {splitting:.4f} vs. 2g = {expected_splitting:.4f} (PASS: < 0.1%)")
print("=" * 70)



---

## Summary & Next Steps

### Key Takeaways from Notebook 4.6
1. **Bridging Discrete and Propagating Photons:** The Gardiner-Collett relation $\hat{a}_{\text{out}}(t) = \hat{a}_{\text{in}}(t) + \sqrt{\kappa}\hat{a}(t)$ connects internal cavity operators to physical traveling-wave inputs and outputs.
2. **Impedance Matching at Critical Coupling:** When external coupling matches internal loss ($\kappa_c = \kappa_i$), destructive interference completely cancels reflection on resonance ($R(\omega_c) = 0$), transferring $100\%$ of incident energy into the cavity mode.
3. **Smith-Chart Topological Phase Slips:** Sweeping probe frequency through resonance causes the reflection vector $r(\omega)$ to trace a circle on the complex plane; in the overcoupled regime ($\kappa_c > \kappa_i$), the origin is enclosed, producing a full $2\pi$ phase slip.
4. **Vacuum Rabi Dressed Transmission:** Strongly coupling an atom to the cavity mode splits the transmission spectrum into a normal-mode doublet separated by $\Delta\omega = 2g$.

---

### Up Next: Notebook 4.7
In [Notebook 4.7: Laser Theory, Micromaser & Trapping States](07_laser_theory_and_micromaser.ipynb), we culminate Module 4 with **fully quantized laser theory**, analyzing the Scully-Lamb master equation, photon statistics across the lasing threshold, and non-classical sub-Poissonian trapping states in the micromaser.

